In [ ]:
import json
from pathlib import Path as _Path
from IPython.display import display, Markdown

_stats_path = _Path.cwd().parent / 'data' / 'processed' / 'stats.json'
with open(_stats_path) as _f:
    oss_stats = json.load(_f)

display(Markdown(f"""
# OSS Pulse: A Data-Driven Story of Open Source PR Activity

**{oss_stats['dataset']['total_prs']:,} pull requests. {oss_stats['dataset']['n_repos']} repositories. {oss_stats['dataset']['n_contributors']:,}+ contributors. 10 years of data.**

This notebook tells a narrative about how open source software is built, maintained, and evolved,
following the data from broad patterns to specific discoveries. Each finding motivates the next question.
"""))

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure oss_pulse is importable
project_root = Path.cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from oss_pulse.visualize.style import PALETTE, save_fig, setup_style

setup_style()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

%matplotlib inline

DATA = project_root / "data"
print(f"Project root: {project_root}")

import json
with open(DATA / "processed/stats.json") as f:
    oss_stats = json.load(f)

In [ ]:
display(Markdown(f"""
---
## 1. \"{oss_stats['dataset']['total_prs']:,} Pull Requests Walk Into a Dataset\"

Before we can tell any story, we need to meet the cast. Let's load the data and get a sense of its shape:
how many repos, how many PRs, what time period, and what languages are represented.
"""))

In [ ]:
# Load all datasets
pr_df = pd.read_parquet(DATA / "processed/pr_events_featured.parquet")
repo_monthly = pd.read_parquet(DATA / "processed/repo_monthly.parquet")
repo_weekly = pd.read_parquet(DATA / "processed/repo_weekly.parquet")
top_repos = pd.read_parquet(DATA / "raw/top_repos.parquet")

# Merge language and stars into PR data
repo_meta = top_repos[["repo_name", "language", "stars"]].copy()
pr_df = pr_df.merge(repo_meta, on="repo_name", how="left")

print(f"Total PRs:           {len(pr_df):,}")
print(f"Repositories:        {pr_df['repo_name'].nunique()}")
print(f"Unique authors:      {pr_df['author'].nunique():,}")
print(f"Date range:          {pr_df['pr_created_at'].min().strftime('%Y-%m-%d')} to {pr_df['pr_created_at'].max().strftime('%Y-%m-%d')}")
print(f"Languages:           {pr_df['language'].nunique()} distinct (+ some repos without a primary language)")

In [ ]:
# Top 10 repos by PR count
top10 = pr_df.groupby("repo_name").size().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    top10.index[::-1],
    top10.values[::-1],
    color=PALETTE["primary"],
    edgecolor="white",
)
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(
        bar.get_width() + 100,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        va="center",
        fontsize=10,
    )
ax.set_xlabel("Number of PRs")
ax.set_title("Top 10 Repositories by PR Volume", fontsize=14, fontweight="bold")
save_fig(fig, "01_top10_repos_by_pr_count")
plt.show()

In [ ]:
# PR outcome distribution
outcome_counts = pr_df["pr_outcome"].value_counts()
outcome_pcts = (outcome_counts / len(pr_df) * 100).round(1)

colors = {
    "merged": PALETTE["success"],
    "closed": PALETTE["warning"],
    "abandoned": PALETTE["accent"],
    "open": PALETTE["primary"],
}
order = ["merged", "closed", "abandoned", "open"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    order,
    [outcome_counts[o] for o in order],
    color=[colors[o] for o in order],
    edgecolor="white",
)
for bar, o in zip(bars, order):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f"{outcome_pcts[o]}%",
        ha="center",
        fontsize=12,
        fontweight="bold",
    )
ax.set_ylabel("Count")
ax.set_title("PR Outcome Distribution", fontsize=14, fontweight="bold")
save_fig(fig, "01_pr_outcome_distribution")
plt.show()

print(f"\nOverall merge rate: {outcome_pcts['merged']}%")
print(f"But the repo-level merge rate ranges from "
      f"{repo_monthly.groupby('repo_name')['merge_rate'].mean().min():.1%} to "
      f"{repo_monthly.groupby('repo_name')['merge_rate'].mean().max():.1%}")

In [ ]:
display(Markdown(f"""
**Key takeaway:** About {oss_stats['outcomes']['merge_rate']:.0f}% of all PRs eventually get merged, but the variance across repositories is enormous.
Some projects merge nearly everything; others reject (or ignore) the majority. This raises a natural question:
is the overall volume of open source contributions actually growing, or are we just seeing the same projects churn?
"""))

In [ ]:
display(Markdown(f"""
---
## 2. \"The Growth Curve No One Talks About\"

Open source is often described as \"exploding\" in popularity, but is it? Let's look at monthly PR volume
across all {oss_stats['dataset']['n_repos']} repos and decompose the signal into trend, seasonality, and residuals.
"""))

In [ ]:
from oss_pulse.analyze.seasonal import stl_decompose, test_stationarity

# Aggregate monthly PR volume
monthly_agg = repo_monthly.groupby(["year", "month"])["pr_count"].sum().reset_index()
monthly_agg["date"] = pd.to_datetime(
    monthly_agg["year"].astype(str) + "-" + monthly_agg["month"].astype(str).str.zfill(2) + "-01"
)
monthly_agg = monthly_agg.sort_values("date")

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(monthly_agg["date"], monthly_agg["pr_count"], color=PALETTE["primary"], linewidth=1.8)
ax.fill_between(monthly_agg["date"], monthly_agg["pr_count"], alpha=0.1, color=PALETTE["primary"])
ax.set_xlabel("Date")
ax.set_ylabel("Monthly PR Count (all repos)")
ax.set_title("Aggregate Monthly PR Volume (2016 - 2026)", fontsize=14, fontweight="bold")
save_fig(fig, "02_monthly_pr_volume_trend")
plt.show()

early_mean = monthly_agg[monthly_agg["date"] < "2017-01-01"]["pr_count"].mean()
recent_mean = monthly_agg[
    (monthly_agg["date"] >= "2025-01-01") & (monthly_agg["date"] < "2026-01-01")
]["pr_count"].mean()
print(f"Average monthly PRs in 2016: {early_mean:,.0f}")
print(f"Average monthly PRs in 2025: {recent_mean:,.0f}")
print(f"Growth factor: {recent_mean / early_mean:.1f}x")

In [ ]:
# STL decomposition on aggregate monthly series
from oss_pulse.visualize.timeseries import plot_decomposition

series_monthly = monthly_agg.set_index("date")["pr_count"]
series_monthly.index = pd.DatetimeIndex(series_monthly.index, freq="MS")

stl_result = stl_decompose(series_monthly, period=12)
fig = plot_decomposition(stl_result, title="STL Decomposition of Aggregate Monthly PR Volume")
save_fig(fig, "02_stl_decomposition")
plt.show()

In [ ]:
# Stationarity tests
stationarity = test_stationarity(series_monthly)
print("Stationarity test results:")
print(f"  ADF statistic: {stationarity['adf_stat']:.4f} (p = {stationarity['adf_pvalue']:.4f})")
print(f"  KPSS statistic: {stationarity['kpss_stat']:.4f} (p = {stationarity['kpss_pvalue']:.4f})")
print(f"  Is stationary: {stationarity['is_stationary']}")
print()
if not stationarity["is_stationary"]:
    print("The series is non-stationary, confirming the visual upward trend is real, not noise.")

In [ ]:
# Overlay top 5 individual repo trends
repo_monthly_meta = repo_monthly.merge(repo_meta, on="repo_name", how="left")
top5_repos = pr_df.groupby("repo_name").size().sort_values(ascending=False).head(5).index.tolist()

fig, ax = plt.subplots(figsize=(16, 6))
cycle_colors = [PALETTE["primary"], PALETTE["accent"], PALETTE["success"], PALETTE["warning"], PALETTE["secondary"]]

for i, repo in enumerate(top5_repos):
    rdata = repo_monthly[repo_monthly["repo_name"] == repo].copy()
    rdata["date"] = pd.to_datetime(
        rdata["year"].astype(str) + "-" + rdata["month"].astype(str).str.zfill(2) + "-01"
    )
    rdata = rdata.sort_values("date")
    short_name = repo.split("/")[-1]
    ax.plot(rdata["date"], rdata["pr_count"], color=cycle_colors[i], linewidth=1.5, label=short_name)

ax.set_xlabel("Date")
ax.set_ylabel("Monthly PR Count")
ax.set_title("Monthly PR Volume: Top 5 Repos", fontsize=14, fontweight="bold")
ax.legend()
save_fig(fig, "02_top5_repo_trends")
plt.show()

print("The aggregate growth is not uniform. A few high-activity repos dominate the curve.")

In [ ]:
_growth_factor = (
    oss_stats['tool_eras']['agentic']['prs_per_month']
    / oss_stats['tool_eras']['pre_copilot']['prs_per_month']
)
display(Markdown(f"""
**Discovery:** OSS contributions have grown roughly {_growth_factor:.1f}x from the pre-Copilot era to today, but the growth is not uniform.
The aggregate trend is dominated by a handful of extremely active repos, while many others stay relatively flat.
The STL decomposition reveals a clear seasonal dip every December/January and the ADF test confirms
the trend is genuine (the series is non-stationary).

But if contributions are clustered in time, are they also clustered in the *day and hour* they happen?
"""))

---
## 3. "When Does the World Write Code?"

We have the hour and day-of-week for every PR. Let's see when open source work actually happens.

In [ ]:
from oss_pulse.visualize.heatmaps import plot_activity_heatmap

fig = plot_activity_heatmap(pr_df, title="When Are PRs Created? (All Repos, UTC)")
save_fig(fig, "03_activity_heatmap_all")
plt.show()

# Find the peak cell
pivot = pr_df.pivot_table(index="day_of_week", columns="hour", aggfunc="size", fill_value=0)
peak = pivot.stack().idxmax()
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(f"Peak activity: {day_names[peak[0]]} at {peak[1]:02d}:00 UTC")
print(f"Weekend contribution rate: {pr_df['is_weekend'].mean():.1%}")

In [ ]:
# Compare patterns by top languages
top_langs = ["Python", "JavaScript", "TypeScript", "Go", "Java"]
lang_df = pr_df[pr_df["language"].isin(top_langs)]

fig, axes = plt.subplots(1, len(top_langs), figsize=(24, 4), sharey=True)
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

for ax, lang in zip(axes, top_langs):
    sub = lang_df[lang_df["language"] == lang]
    pivot = sub.pivot_table(index="day_of_week", columns="hour", aggfunc="size", fill_value=0)
    pivot = pivot.reindex(index=range(7), columns=range(24), fill_value=0)
    sns.heatmap(pivot, ax=ax, cmap="Blues", cbar=False, linewidths=0.3)
    ax.set_title(lang, fontsize=12, fontweight="bold")
    ax.set_yticklabels(day_labels, rotation=0)
    if ax != axes[0]:
        ax.set_ylabel("")
    ax.set_xlabel("Hour")

fig.suptitle("Activity Patterns by Language (UTC)", fontsize=14, fontweight="bold", y=1.05)
fig.tight_layout()
save_fig(fig, "03_activity_heatmap_by_language")
plt.show()

In [ ]:
# Weekend rates by language
weekend_by_lang = (
    lang_df.groupby("language")["is_weekend"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(1)
)
print("Weekend contribution rate by language:")
for lang, pct in weekend_by_lang.items():
    print(f"  {lang}: {pct}%")

In [ ]:
# Weekend rate: computed from unique PRs (not events)
_pr_unique = pr_df.drop_duplicates(subset=['repo_name', 'pr_number'])
_weekend_rate_pct = _pr_unique['is_weekend'].mean() * 100

display(Markdown(f"""
**Discovery:** The workweek pattern is unmistakable. The vast majority of PRs are created Monday through Friday,
peaking during afternoon UTC hours (which corresponds to morning in the US and afternoon in Europe).
Only about {_weekend_rate_pct:.0f}% of contributions happen on weekends.

This strongly suggests that most open source work happens during business hours, probably as part of
professional roles rather than purely as a hobby. The language-level patterns are broadly similar, though
the exact peak hours shift slightly by ecosystem.

Now that we know *when* code is contributed, the next question is: how quickly does it get reviewed and merged?
"""))

In [ ]:
display(Markdown(f"""
---
## 4. \"The {oss_stats['merge_time']['median_hours']:.1f}-Hour PR: How Fast Is Open Source?\"

Speed matters. A PR that sits for weeks discourages future contributions. Let's look at the distribution
of merge times and see what factors accelerate or slow down the process.
"""))

In [ ]:
from oss_pulse.analyze.response_time import merge_time_by_segment

merged_prs = pr_df[pr_df["time_to_merge_hours"].notna()].copy()

# Merge time distribution (log scale)
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(
    merged_prs["time_to_merge_hours"].clip(upper=2000),
    bins=100,
    color=PALETTE["primary"],
    edgecolor="white",
    alpha=0.85,
)
ax.set_yscale("log")
median_h = merged_prs["time_to_merge_hours"].median()
ax.axvline(median_h, color=PALETTE["accent"], linestyle="--", linewidth=1.5, label=f"Median: {median_h:.1f}h")
ax.set_xlabel("Hours to Merge")
ax.set_ylabel("Count (log scale)")
ax.set_title("Distribution of Time to Merge", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
save_fig(fig, "04_merge_time_distribution")
plt.show()

p95 = merged_prs["time_to_merge_hours"].quantile(0.95)
print(f"Median time to merge: {median_h:.1f} hours")
print(f"95th percentile: {p95:.0f} hours ({p95/24:.0f} days)")
print(f"This means half of all PRs are merged within ~{median_h:.0f} hours,")
print(f"but the slowest 5% take over a month.")

In [ ]:
from oss_pulse.visualize.comparison import plot_boxplot_comparison

# Merge time by PR size (cap at 500h for readability)
capped = merged_prs[merged_prs["time_to_merge_hours"] < 500].copy()

fig = plot_boxplot_comparison(
    capped, metric="time_to_merge_hours", group="pr_size_bucket",
    title="Merge Time by PR Size (capped at 500h)",
)
save_fig(fig, "04_merge_time_by_size")
plt.show()

size_stats = merge_time_by_segment(merged_prs, "pr_size_bucket")
print("Median merge time by PR size:")
for _, row in size_stats.iterrows():
    print(f"  {row['pr_size_bucket']:>8s}: {row['median_hours']:.1f} hours")

In [ ]:
# Merge time by author type
human_merged = capped[~capped["is_bot"]].copy()

fig = plot_boxplot_comparison(
    human_merged, metric="time_to_merge_hours", group="author_class",
    title="Merge Time by Author Type (non-bots, capped at 500h)",
)
save_fig(fig, "04_merge_time_by_author_type")
plt.show()

author_stats = merge_time_by_segment(pr_df[~pr_df["is_bot"]], "author_class")
print("Median merge time by author class:")
for _, row in author_stats.iterrows():
    print(f"  {row['author_class']:>12s}: {row['median_hours']:.1f} hours")

In [ ]:
# Has merge time improved over the years? Monthly median trend
merged_prs_ts = merged_prs.copy()
merged_prs_ts["year_month"] = merged_prs_ts["pr_merged_at"].dt.to_period("M")
monthly_median = merged_prs_ts.groupby("year_month")["time_to_merge_hours"].median()
monthly_median.index = monthly_median.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_median.index, monthly_median.values, color=PALETTE["primary"], linewidth=1.5)
ax.set_xlabel("Date")
ax.set_ylabel("Median Hours to Merge")
ax.set_title("Monthly Median Merge Time (All Repos)", fontsize=14, fontweight="bold")
save_fig(fig, "04_merge_time_trend")
plt.show()

In [ ]:
display(Markdown(f"""
**Discovery:** The median PR takes about {oss_stats['merge_time']['median_hours']:.1f} hours to merge, but first-timers wait significantly longer
than maintainers. Large PRs also take longer, which is expected given the review burden.

Is the longer wait for first-timers gatekeeping, or is it healthy quality control?
One way to investigate is to look at what happens *after* the first PR.
If contributors feel welcomed, they come back. If they don't, the project has a retention problem.
"""))

In [ ]:
_dropout_pct = 100 - oss_stats['funnel']['2nd_pr_pct']
display(Markdown(f"""
---
## 5. \"{_dropout_pct:.0f}% Never Come Back\"

Every open source project depends on new contributors. But how many people who open their first PR
ever come back for a second? Let's trace the contributor funnel.
"""))

In [ ]:
from oss_pulse.analyze.funnel import build_contributor_funnel, compute_retention_rates, funnel_by_segment
from oss_pulse.visualize.comparison import plot_funnel

funnel = build_contributor_funnel(pr_df)
funnel = compute_retention_rates(funnel)

print("Contributor Engagement Funnel:")
print(funnel.to_string(index=False))
print()

first_pr_count = funnel.loc[funnel["stage"] == "1st_pr", "count"].values[0]
second_pr_count = funnel.loc[funnel["stage"] == "2nd_pr", "count"].values[0]
dropout_pct = (1 - second_pr_count / first_pr_count) * 100
print(f"{first_pr_count:,} people opened at least one PR.")
print(f"Only {second_pr_count:,} came back for a second ({dropout_pct:.0f}% dropout at the first transition).")
print(f"Only {funnel.loc[funnel['stage'] == 'regular', 'count'].values[0]} reached 'regular' status (20+ PRs).")

In [ ]:
fig = plot_funnel(funnel, title=f"Contributor Engagement Funnel ({oss_stats['dataset']['n_contributors']:,}+ authors)")
save_fig(fig, "05_contributor_funnel")
plt.show()

In [ ]:
# Funnel by language
pr_with_lang = pr_df[pr_df["language"].isin(["Python", "JavaScript", "TypeScript", "Go", "Java"])].copy()
lang_funnel = funnel_by_segment(pr_with_lang, "language")

# Compare 1st -> 2nd PR retention by language
retention_comparison = []
for lang in pr_with_lang["language"].unique():
    lf = lang_funnel[lang_funnel["language"] == lang]
    if len(lf) >= 2:
        first = lf.loc[lf["stage"] == "1st_pr", "count"].values[0]
        second = lf.loc[lf["stage"] == "2nd_pr", "count"].values[0]
        retention_comparison.append({
            "language": lang,
            "first_pr_authors": first,
            "second_pr_authors": second,
            "retention_1to2": second / first * 100 if first > 0 else 0,
        })

ret_df = pd.DataFrame(retention_comparison).sort_values("retention_1to2", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(ret_df["language"], ret_df["retention_1to2"], color=PALETTE["primary"], edgecolor="white")
for i, (_, row) in enumerate(ret_df.iterrows()):
    ax.text(
        row["retention_1to2"] + 0.5,
        i,
        f"{row['retention_1to2']:.1f}%",
        va="center",
        fontsize=10,
    )
ax.set_xlabel("1st-to-2nd PR Retention Rate (%)")
ax.set_title("Contributor Retention by Language Ecosystem", fontsize=14, fontweight="bold")
save_fig(fig, "05_retention_by_language")
plt.show()

In [ ]:
# Retention by era: has it improved over time?
pr_human = pr_df[~pr_df["is_bot"]].copy()
pr_human["era"] = pd.cut(
    pr_human["pr_created_at"].dt.year,
    bins=[2015, 2018, 2021, 2024, 2027],
    labels=["2016-2018", "2019-2021", "2022-2024", "2025-2026"],
)
era_funnel = funnel_by_segment(pr_human.dropna(subset=["era"]), "era")

era_retention = []
for era in era_funnel["era"].unique():
    ef = era_funnel[era_funnel["era"] == era]
    first = ef.loc[ef["stage"] == "1st_pr", "count"].values[0] if len(ef[ef["stage"] == "1st_pr"]) > 0 else 0
    second = ef.loc[ef["stage"] == "2nd_pr", "count"].values[0] if len(ef[ef["stage"] == "2nd_pr"]) > 0 else 0
    era_retention.append({"era": era, "retention_pct": second / first * 100 if first > 0 else 0})

era_ret_df = pd.DataFrame(era_retention)
print("1st-to-2nd PR retention by era:")
for _, row in era_ret_df.iterrows():
    print(f"  {row['era']}: {row['retention_pct']:.1f}%")

In [ ]:
_dropout_pct = 100 - oss_stats['funnel']['2nd_pr_pct']
display(Markdown(f"""
**Discovery:** Open source has a massive attrition problem. Around {_dropout_pct:.0f}% of contributors never return after
their first PR. The funnel narrows dramatically at every stage, from {oss_stats['funnel']['total']:,} first-time contributors down to just
{oss_stats['funnel']['regular']:,} who reach \"regular\" status ({oss_stats['funnel']['regular_pct']:.0f}% of starters, defined as 20+ PRs).

The first-to-second PR transition is where the bulk of contributors are lost, and this pattern holds
across all language ecosystems. The question of *why* they leave is harder to answer with PR data alone,
but one contributing factor might be how the broader ecosystem is evolving.

Speaking of ecosystem-level shifts, has the rise of AI coding tools like GitHub Copilot and ChatGPT
left any detectable fingerprint on PR activity?
"""))

---
## 6. "Did AI Change Everything?"

Rather than starting with the assumption that AI tools changed contribution patterns, let's let the data
tell us. We'll run unsupervised changepoint detection on the aggregate weekly PR volume and see
if any structural breaks coincide with major AI milestones.

In [ ]:
from oss_pulse.analyze.changepoint import detect_changepoints, compute_pre_post_stats

# Build aggregate weekly series with proper datetime index
weekly_agg = repo_weekly.groupby("year_week")["pr_count"].sum().sort_index()

# Parse ISO week strings to dates (handle edge cases like W53)
def parse_iso_week(w):
    try:
        return pd.to_datetime(w + "-1", format="%G-W%V-%u")
    except Exception:
        return pd.NaT

week_dates = weekly_agg.index.map(parse_iso_week)
weekly_series = pd.Series(weekly_agg.values, index=week_dates, name="pr_count").dropna()

# Detect changepoints (unsupervised - no assumed dates)
changepoints = detect_changepoints(weekly_series, method="pelt", min_size=12)
print(f"Changepoints detected: {len(changepoints)} at indices {changepoints}")

# Plot the series with changepoint markers
setup_style()
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(weekly_series.index, weekly_series.values, color=PALETTE["primary"], linewidth=0.8, alpha=0.7)

# Overlay a rolling mean for clarity
rolling = weekly_series.rolling(8).mean()
ax.plot(rolling.index, rolling.values, color=PALETTE["primary"], linewidth=2, label="8-week rolling mean")

# Mark changepoints
events_df = pd.read_csv("../config/events.csv")
ai_events = events_df[events_df["category"] == "ai"]
for cp_idx in changepoints:
    cp_date = weekly_series.index[cp_idx]
    ax.axvline(cp_date, color=PALETTE["accent"], linestyle="--", alpha=0.7, linewidth=1)

# Mark known AI events
for _, event in ai_events.iterrows():
    ev_date = pd.to_datetime(event["date"])
    if ev_date >= weekly_series.index.min() and ev_date <= weekly_series.index.max():
        ax.axvline(ev_date, color=PALETTE["success"], linestyle=":", alpha=0.5, linewidth=1)
        ax.text(ev_date, ax.get_ylim()[1] * 0.95, event["event"], rotation=45, fontsize=7,
                ha="right", va="top", color=PALETTE["success"])

ax.set_title("Weekly PR Volume with Detected Changepoints", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("PRs per week")
ax.legend(loc="upper left")
save_fig(fig, "06_changepoints")
plt.show()



In [ ]:
# Pre/post analysis for the most significant changepoint
if changepoints:
    best_cp_idx = changepoints[0]
    best_effect = 0
    best_stats = None
    
    for cp_idx in changepoints:
        stats = compute_pre_post_stats(weekly_series, cp_idx)
        if abs(stats["effect_size"]) > abs(best_effect):
            best_effect = stats["effect_size"]
            best_cp_idx = cp_idx
            best_stats = stats
    
    cp_date = weekly_series.index[best_cp_idx]
    print(f"Most significant changepoint: {cp_date.strftime('%Y-%m-%d')}")
    print(f"  Pre:  mean={best_stats['pre_mean']:.1f}, std={best_stats['pre_std']:.1f} ({best_stats['pre_n']} weeks)")
    print(f"  Post: mean={best_stats['post_mean']:.1f}, std={best_stats['post_std']:.1f} ({best_stats['post_n']} weeks)")
    print(f"  Cohen's d: {best_stats['effect_size']:.3f}")
else:
    print("No changepoints detected")



In [ ]:
display(Markdown(f"""
**Discovery:** The changepoint detector finds structural breaks in the PR volume series, and we can compare
their timing against major AI tool launches. Whether any detected shift *caused by* AI tools versus other
factors (organic growth, new repos entering the dataset, etc.) remains an open question.
The data reveals correlation at best, not causation.

Rather than speculate further on macro trends, let's turn inward and ask: across all {oss_stats['dataset']['n_repos']} repos,
which ones are actually *healthy* projects?
"""))

---
## 7. "The Health Checkup: Which Projects Are Thriving?"

We define "health" as a weighted composite of five signals: response time, merge rate, contributor diversity,
recent trend, and bus factor. Let's compute the index and see who scores well and who doesn't.

In [ ]:
from oss_pulse.analyze.health_index import compute_health_components, compute_health_index, rank_repos

components = compute_health_components(repo_monthly)
components["health_index"] = compute_health_index(components)
ranked = rank_repos(components)

# Merge stars for later scatter
ranked = ranked.merge(repo_meta[["repo_name", "stars"]], on="repo_name", how="left")

print(f"Health index computed for {len(ranked)} repos.")
print(f"Range: {ranked['health_index'].min():.1f} to {ranked['health_index'].max():.1f}")
print(f"Median: {ranked['health_index'].median():.1f}")

In [ ]:
# Top 15 and Bottom 15
top15 = ranked.head(15).sort_values("health_index")
bottom15 = ranked.tail(15).sort_values("health_index")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

ax1.barh(top15["repo_name"], top15["health_index"], color=PALETTE["success"], edgecolor="white")
ax1.set_xlabel("Health Index")
ax1.set_title("Top 15 Healthiest Repos", fontsize=13, fontweight="bold")
for i, (_, row) in enumerate(top15.iterrows()):
    ax1.text(row["health_index"] + 0.5, i, f"{row['health_index']:.1f}", va="center", fontsize=9)

ax2.barh(bottom15["repo_name"], bottom15["health_index"], color=PALETTE["accent"], edgecolor="white")
ax2.set_xlabel("Health Index")
ax2.set_title("Bottom 15 by Health Index", fontsize=13, fontweight="bold")
for i, (_, row) in enumerate(bottom15.iterrows()):
    ax2.text(row["health_index"] + 0.5, i, f"{row['health_index']:.1f}", va="center", fontsize=9)

fig.tight_layout()
save_fig(fig, "07_health_top_bottom_15")
plt.show()

In [ ]:
# Radar / spider chart: component profiles of top 3 vs bottom 3
from math import pi

score_cols = [
    "response_time_score", "merge_rate_score", "diversity_score",
    "trend_score", "bus_factor_score",
]
labels = ["Response\nTime", "Merge\nRate", "Diversity", "Trend", "Bus\nFactor"]
N = len(labels)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # close the polygon

top3 = ranked.head(3)
bot3 = ranked.tail(3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": "polar"})

for ax, group, title_text, palette_key in [
    (ax1, top3, "Top 3", [PALETTE["primary"], PALETTE["success"], PALETTE["warning"]]),
    (ax2, bot3, "Bottom 3", [PALETTE["accent"], PALETTE["secondary"], "#9333ea"]),
]:
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 100)

    for i, (_, row) in enumerate(group.iterrows()):
        values = [row[c] for c in score_cols]
        values += values[:1]
        short = row["repo_name"].split("/")[-1]
        ax.plot(angles, values, linewidth=1.5, color=palette_key[i], label=short)
        ax.fill(angles, values, alpha=0.1, color=palette_key[i])

    ax.set_title(title_text, fontsize=13, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)

fig.tight_layout()
save_fig(fig, "07_health_radar_comparison")
plt.show()

In [ ]:
# Scatter: health index vs stars
plot_df = ranked.dropna(subset=["stars"]).copy()

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(
    plot_df["stars"],
    plot_df["health_index"],
    s=50,
    color=PALETTE["primary"],
    alpha=0.7,
    edgecolors="white",
    linewidth=0.5,
)

# Label outliers
for _, row in plot_df.iterrows():
    if row["health_index"] > plot_df["health_index"].quantile(0.9) or \
       row["health_index"] < plot_df["health_index"].quantile(0.1) or \
       row["stars"] > plot_df["stars"].quantile(0.9):
        ax.annotate(
            row["repo_name"].split("/")[-1],
            (row["stars"], row["health_index"]),
            fontsize=7,
            alpha=0.8,
            xytext=(5, 5),
            textcoords="offset points",
        )

ax.set_xlabel("GitHub Stars")
ax.set_ylabel("Health Index")
ax.set_title("Health Index vs. GitHub Stars", fontsize=14, fontweight="bold")

corr = plot_df[["stars", "health_index"]].corr().iloc[0, 1]
ax.text(
    0.05, 0.95,
    f"Pearson r = {corr:.2f}",
    transform=ax.transAxes,
    fontsize=12,
    va="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
)
save_fig(fig, "07_health_vs_stars")
plt.show()

print(f"Correlation between stars and health index: {corr:.2f}")
if abs(corr) < 0.3:
    print("Popularity (stars) is only weakly correlated with project health.")
    print("Some of the most starred repos score poorly on health metrics.")

**Discovery:** The health index reveals a wide range of project vitality. The radar charts make it clear
that healthy projects tend to score well across multiple dimensions simultaneously, while struggling projects
often have a single failing component (e.g., declining trend or slow response times) that drags down the whole score.

Most importantly, star count is a weak predictor of health. Some of the most popular repos have
mediocre health scores, while some lesser-known projects score highly. Stars measure attention,
not sustainability.

For our final analysis, let's look at the lifecycle of individual PRs: how long do they survive
before being resolved, and what predicts whether they'll be abandoned?

---
## 8. "Predicting Decline: The Early Warning Signs"

Using survival analysis, we can model the probability that a PR remains unresolved over time.
Kaplan-Meier curves show us how quickly PRs get resolved, and whether the pattern differs
by PR size or author type.

In [ ]:
from oss_pulse.analyze.abandonment import build_survival_data, fit_kaplan_meier
from oss_pulse.visualize.comparison import plot_survival_curves

surv_data = build_survival_data(pr_df)

# Overall KM curve
kmf_all = fit_kaplan_meier(surv_data)
median_surv = kmf_all.median_survival_time_

print(f"Survival data: {len(surv_data):,} PRs")
print(f"Median survival time: {median_surv:.1f} days")
print()
print("Survival probabilities at key timepoints:")
for days in [1, 7, 30, 90, 180, 365]:
    prob = kmf_all.predict(days)
    print(f"  {days:>3d} days: {prob:.1%} of PRs still unresolved")

In [ ]:
# KM curves split by PR size
from lifelines import KaplanMeierFitter

size_curves = []
for size in ["small", "medium", "large"]:
    subset = surv_data[surv_data["pr_size_bucket"] == size]
    kmf = KaplanMeierFitter()
    kmf.fit(subset["duration_days"], subset["event"], label=size)
    size_curves.append((kmf, size))

fig = plot_survival_curves(size_curves, title="PR Survival by Size")
plt.xlabel("Days")
save_fig(fig, "08_survival_by_size")
plt.show()

In [ ]:
# KM curves split by author type (exclude bots)
author_curves = []
for atype in ["maintainer", "regular", "first-timer"]:
    subset = surv_data[surv_data["author_type"] == atype]
    if len(subset) < 10:
        continue
    kmf = KaplanMeierFitter()
    kmf.fit(subset["duration_days"], subset["event"], label=atype)
    author_curves.append((kmf, atype))

fig = plot_survival_curves(author_curves, title="PR Survival by Author Type")
plt.xlabel("Days")
save_fig(fig, "08_survival_by_author_type")
plt.show()

# Print median survival times
print("Median survival time by author type:")
for kmf, label in author_curves:
    print(f"  {label:>12s}: {kmf.median_survival_time_:.1f} days")

**Discovery:** The survival curves confirm what the merge time analysis hinted at.
Most PRs are resolved quickly, but there's a long tail of PRs that linger for months.
Maintainer PRs get resolved fastest (they often self-merge), while first-timer PRs take longer.
Large PRs also survive longer, likely because they require more extensive review.

The Kaplan-Meier model treats every unresolved PR as "censored" (still at risk),
which gives us a more honest picture than raw averages would.

In [ ]:
_maint_gain = oss_stats['productivity_by_author_type']['maintainer']['productivity_change_pct']
display(Markdown(f"""
---
## 8b. \"Who's Actually More Productive?\"

The overall productivity increase (driven by a {_maint_gain:.0f}% surge among maintainers) masks very different dynamics across contributor types.
Let's break it down: are first-timers, regulars, and maintainers all producing more? And crucially,
are they getting their PRs merged at the same rate?
"""))

In [ ]:
from oss_pulse.analyze.productivity import (
    compute_productivity_timeseries,
    compute_merge_rate_timeseries,
)
from oss_pulse.visualize.timeseries import plot_grouped_trend

prod_ts = compute_productivity_timeseries(pr_df)
merge_ts = compute_merge_rate_timeseries(pr_df)

fig = plot_grouped_trend(
    prod_ts, date_col="year_month", value_col="prs_per_person",
    group_col="author_class",
    title="PRs per Contributor per Month by Author Type",
    ylabel="PRs / Person / Month",
)
save_fig(fig, "ai_07_productivity_by_author_type")
plt.show()

In [ ]:
fig = plot_grouped_trend(
    merge_ts, date_col="year_month", value_col="merge_rate",
    group_col="author_class",
    title="Merge Rate by Author Type",
    ylabel="Merge Rate",
    ylim=(0, 1),
)
save_fig(fig, "ai_08_merge_rate_by_author_type")
plt.show()

In [ ]:
_p = oss_stats['productivity_by_author_type']
_m_pre  = _p['maintainer']['pre_2023']['prs_per_person_month']
_m_post = _p['maintainer']['post_2023']['prs_per_person_month']
_m_gain = _p['maintainer']['productivity_change_pct']
_r_gain = _p['regular']['productivity_change_pct']
_ft_mr_pre  = _p['first-timer']['pre_2023']['merge_rate_pct']
_ft_mr_post = _p['first-timer']['post_2023']['merge_rate_pct']
_reg_mr_pre  = _p['regular']['pre_2023']['merge_rate_pct']
_reg_mr_post = _p['regular']['post_2023']['merge_rate_pct']
_maint_mr_pre  = _p['maintainer']['pre_2023']['merge_rate_pct']
_maint_mr_post = _p['maintainer']['post_2023']['merge_rate_pct']

display(Markdown(f"""
**Discovery:** The {_m_gain:.0f}% productivity increase among maintainers ({_m_pre} to {_m_post} PRs/month) is the dominant story.
First-timers stay at 1.0 by definition, and regulars barely moved (+{_r_gain:.0f}%). Meanwhile, merge rates
dropped sharply for first-timers ({_ft_mr_pre:.0f}% to {_ft_mr_post:.0f}%) and regulars ({_reg_mr_pre:.0f}% to {_reg_mr_post:.0f}%), while maintainers held steady
at ~{_maint_mr_post:.0f}%. More people are producing more code, but the gatekeeping has tightened for everyone except
the inner circle.
"""))

In [ ]:
# ── Section 9b: A First-Timer's Guide ───────────────────────────────

_ft = oss_stats.get('first_timer_trees', {})
_ft_langs = [lang for lang in ['Python', 'TypeScript', 'Go', 'Rust', 'C++', 'Java', 'C', 'Ruby', 'C#', 'JavaScript'] if lang in _ft]

if _ft_langs:
    # Build summary table
    _rows = []
    for lang in _ft_langs:
        d = _ft[lang]
        top_feat = d['feature_importances'][0]['feature'] if d['feature_importances'] else 'N/A'
        top_imp  = d['feature_importances'][0]['importance'] if d['feature_importances'] else 0
        top_repo = d['recommended_repos'][0]['repo'] if d['recommended_repos'] else 'N/A'
        top_rate = d['recommended_repos'][0]['ft_merge_rate'] if d['recommended_repos'] else 0
        _rows.append({
            'Language': lang,
            'Samples': f"{d['n_samples']:,}",
            'Accuracy': f"{d['accuracy']:.1%}",
            'AUC': f"{d['auc']:.3f}",
            'Top Feature': f"{top_feat} ({top_imp:.0%})",
            'Best Repo': f"{top_repo} ({top_rate:.0f}%)",
        })
    _ft_table = pd.DataFrame(_rows)
    display(Markdown('---\n## 9a. "A First-Timer\'s Guide to Getting Merged"\n\n'
                     'We trained a shallow decision tree (depth 4) per language to predict whether a first-timer\'s PR gets merged. '
                     'The goal is not perfect prediction but *interpretability*: what should a newcomer focus on?\n'))
    display(_ft_table)
else:
    display(Markdown('*First-timer tree analysis not yet available. Run `python -m oss_pulse.analyze.first_timer_tree`.*'))


In [ ]:
# First-timer recommendations figure
if _ft_langs:
    from oss_pulse.visualize.comparison import plot_first_timer_recommendations
    fig = plot_first_timer_recommendations(_ft)
    save_fig(fig, 'ai_09a_first_timer_recommendations')
    plt.show()


In [ ]:
# Narrative: per-language actionable advice
if _ft_langs:
    # Compute the importance range for repo_merge_rate across all languages
    _rmr_imps = []
    for lang in _ft_langs:
        for fi in _ft[lang]['feature_importances']:
            if fi['feature'] == 'repo_merge_rate':
                _rmr_imps.append(fi['importance'])
    _rmr_min = min(_rmr_imps) if _rmr_imps else 0
    _rmr_max = max(_rmr_imps) if _rmr_imps else 0

    _lines = [
        f'**The single biggest predictor is `repo_merge_rate`**, accounting for '
        f'{_rmr_min:.0%} to {_rmr_max:.0%} of the decision across all languages. '
        f'In other words, picking the right repo matters far more than PR size, timing, or day of week.\n\n',
        'Here is the actionable takeaway per language:\n\n',
    ]

    for lang in _ft_langs:
        d = _ft[lang]
        rule = d['decision_rules']
        recs = d['recommended_repos'][:3]
        rec_strs = []
        for r in recs:
            rec_strs.append(f"**{r['repo']}** ({r['ft_merge_rate']:.0f}% first-timer merge rate, {r['stars']:,} stars)")
        rec_list = ', '.join(rec_strs)
        _lines.append(f'- **{lang}** (n={d["n_samples"]:,}, AUC={d["auc"]:.3f}): '
                      f'Rule: `{rule}`. Top repos: {rec_list}.\n')

    _lines.append(
        f'\n**Bottom line:** If you are a first-timer, look at the repo\'s overall merge rate before investing '
        f'effort. A repo that merges 80%+ of its PRs is far more likely to merge yours than one sitting at 40%.'
    )

    display(Markdown(''.join(_lines)))


---
## 9. "What We Learned"

This analysis traced a path from raw PR data to actionable findings about how open source software
is built, maintained, and evolved. Here are the key takeaways:

In [ ]:
_p = oss_stats['productivity_by_author_type']
_dropout_pct = 100 - oss_stats['funnel']['2nd_pr_pct']
_regular_pct = oss_stats['funnel']['regular_pct']
_growth_factor = (
    oss_stats['tool_eras']['agentic']['prs_per_month']
    / oss_stats['tool_eras']['pre_copilot']['prs_per_month']
)
_pr_unique = pr_df.drop_duplicates(subset=['repo_name', 'pr_number'])
_weekend_rate_pct = _pr_unique['is_weekend'].mean() * 100

display(Markdown(f"""
### Key Findings

1. **The merge rate is {oss_stats['outcomes']['merge_rate']:.0f}%, but the variance is massive.**
   Across {oss_stats['dataset']['n_repos']} repos, some merge nearly every PR while others reject the majority.
   The \"average\" hides significant project-level differences.

2. **PR volume has grown ~{_growth_factor:.1f}x from the pre-Copilot era to today,**
   but the growth is concentrated in a few high-activity repos.
   Many projects maintain a steady, modest pace.

3. **Open source is a workday activity.**
   Only {_weekend_rate_pct:.0f}% of contributions happen on weekends.
   Peak activity is Monday through Thursday, 14:00-17:00 UTC, suggesting most contributors are
   working during business hours in US/European time zones.

4. **The median PR merges in {oss_stats['merge_time']['median_hours']:.1f} hours,**
   but first-timer PRs wait significantly longer than maintainer PRs, and large PRs take longer than small ones.

5. **{_dropout_pct:.0f}% of contributors never come back after their first PR.**
   The first-to-second PR transition is where most contributors are lost.
   Only about {_regular_pct:.1f}% of first-time contributors ever reach \"regular\" status.

6. **Changepoint detection finds structural breaks** in PR volume that may or may not coincide with
   AI tool launches. The data shows correlation, not causation.

7. **Stars are a poor proxy for project health.** The composite health index (response time, merge rate,
   diversity, trend, bus factor) reveals that some of the most popular repos score poorly,
   while lesser-known projects can be very healthy.

8. **Survival analysis confirms the two-speed nature of OSS.** Most PRs resolve within days,
   but a significant tail lingers for months. Author type and PR size are strong predictors
   of resolution speed.

### Limitations

- **{oss_stats['dataset']['n_repos']} repos.** These are among the most popular repos on GitHub,
  not representative of the long tail of smaller projects.
- **Selection bias toward starred repos.** Popularity is the sampling criterion.
- **PR-level data only.** We lack issue data, commit-level data, and discussion/community
  engagement signals that would give a fuller picture.
- **UTC timestamps.** Contributor location inference from UTC activity patterns is approximate at best.

### Future Work

- Expand coverage and include issue data
- Add commit-level analysis to distinguish code contributions from documentation/CI changes
- Build a predictive model for contributor retention
- Track the health index over time to detect declining projects early
"""))

In [ ]:
print("Analysis complete.")
print(f"Total figures saved: check output/figures/ for all .png files.")
print(f"Data sources: {len(pr_df):,} PRs across {pr_df['repo_name'].nunique()} repos")